# Combined Access Score and Network Threshold Analysis

**Tess Vu**

This notebook merges three analytical streams into a single SAL-level dataset:

1. **2SFCA accessibility scores** (`tess_all_access_w_snap.csv`): Ai_walk, Ai_drive, and snap distances.
2. **k-nearest pharmacy distances** (`sal_pharmacy_distances_k3.csv`): Euclidean, walk, and drive for k=1,2,3.
3. **Threshold exceedance flags** (`sal_threshold_flags_k3.csv`): Binary flags at policy-relevant thresholds.

The unified table is exported as both a CSV and a shapefile for downstream mapping and app integration.

## SETUP AND IMPORTS

In [68]:
import pandas as pd
import geopandas as gpd
import numpy as np
import os
import warnings

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")

## CONFIGURATION

In [69]:
# INPUT PATHS
# 2SFCA scores with snap distances appended by snap_distance_append notebook.
ALL_ACCESS_PATH = "data/tess_all_access_w_snap.csv"

# k-nearest pharmacy distances from sal_pharmacy_distance_k3 notebook.
PHARMACY_DIST_PATH = "data/networks/sal_pharmacy_distances_k3.csv"

# Threshold exceedance flags from network_threshold notebook.
THRESHOLD_FLAG_PATH = "data/networks/sal_threshold_flags_k3.csv"

# SAL dedup shapefile for geometry export.
SAL_SHP_PATH = "data/sal_w_ward_dedup/sal_w_ward_dedup.shp"

# OUTPUT PATHS
OUTPUT_DIR = "data/combined"
os.makedirs(OUTPUT_DIR, exist_ok = True)

CSV_OUTPUT = os.path.join(OUTPUT_DIR, "sal_combined_access.csv")
SHP_OUTPUT_DIR = os.path.join(OUTPUT_DIR, "sal_combined_access_shp")
os.makedirs(SHP_OUTPUT_DIR, exist_ok = True)

# POLICY THRESHOLDS FOR ACCESS TYPOLOGY.
# These match the reference thresholds used in the 2SFCA and threshold notebooks.
WALK_THRESHOLD_KM = 3
DRIVE_THRESHOLD_KM = 10

# SNAP DISTANCE FLAG THRESHOLD.
SNAP_FLAG_M = 500

print(f"WALK THRESHOLD: {WALK_THRESHOLD_KM} km")
print(f"DRIVE THRESHOLD: {DRIVE_THRESHOLD_KM} km")
print(f"SNAP FLAG: >{SNAP_FLAG_M} m")

WALK THRESHOLD: 3 km
DRIVE THRESHOLD: 10 km
SNAP FLAG: >500 m


## LOAD SOURCE DATA

In [70]:
# Load the three source tables.
all_access = pd.read_csv(ALL_ACCESS_PATH, low_memory = False)
pharmacy_dist = pd.read_csv(PHARMACY_DIST_PATH)
threshold_flag = pd.read_csv(THRESHOLD_FLAG_PATH)

print(f"ALL_ACCESS: {all_access.shape[0]} rows, {all_access.shape[1]} cols")
print(f"PHARMACY_DIST: {pharmacy_dist.shape[0]} rows, {pharmacy_dist.shape[1]} cols")
print(f"THRESHOLD_FLAG: {threshold_flag.shape[0]} rows, {threshold_flag.shape[1]} cols")

ALL_ACCESS: 38380 rows, 91 cols
PHARMACY_DIST: 38380 rows, 26 cols
THRESHOLD_FLAG: 38380 rows, 51 cols


In [71]:
# Load SAL geometry for shapefile export.
sal_geo = gpd.read_file(SAL_SHP_PATH)
sal_geo["EA_CODE"] = pd.to_numeric(sal_geo["EA_CODE"], errors = "coerce").astype("Int64")

print(f"SAL GEOMETRY: {sal_geo.shape[0]} features, CRS: {sal_geo.crs}")

SAL GEOMETRY: 38380 features, CRS: EPSG:32735


## EA_CODE ALIGNMENT CHECK

In [72]:
# Normalize EA_CODE types across all three tables.
all_access["EA_CODE"] = pd.to_numeric(all_access["EA_CODE"], errors = "coerce").astype("Int64")
pharmacy_dist["EA_CODE"] = pd.to_numeric(pharmacy_dist["EA_CODE"], errors = "coerce").astype("Int64")
threshold_flag["EA_CODE"] = pd.to_numeric(threshold_flag["EA_CODE"], errors = "coerce").astype("Int64")

# Set comparison.
ai_codes = set(all_access["EA_CODE"].dropna())
dist_codes = set(pharmacy_dist["EA_CODE"].dropna())
flag_codes = set(threshold_flag["EA_CODE"].dropna())
geo_codes = set(sal_geo["EA_CODE"].dropna())

all_four = ai_codes & dist_codes & flag_codes & geo_codes

print("EA_CODE ALIGNMENT")
print(f"  all_access:     {len(ai_codes)}")
print(f"  pharmacy_dist:  {len(dist_codes)}")
print(f"  threshold_flag: {len(flag_codes)}")
print(f"  sal_geometry:   {len(geo_codes)}")
print(f"  intersection:   {len(all_four)}")
print()

# Report any orphans.
only_ai = ai_codes - dist_codes
only_dist = dist_codes - ai_codes
only_flag = flag_codes - ai_codes
print(f"  In all_access only:     {len(only_ai)}")
print(f"  In pharmacy_dist only:  {len(only_dist)}")
print(f"  In threshold_flag only: {len(only_flag)}")

EA_CODE ALIGNMENT
  all_access:     38380
  pharmacy_dist:  38380
  threshold_flag: 38380
  sal_geometry:   38380
  intersection:   38380

  In all_access only:     0
  In pharmacy_dist only:  0
  In threshold_flag only: 0


## SNAP DISTANCE ANALYSIS

Before merging, profile the snap distances appended by the snap_distance_append
notebook. Snap distance is the straight-line gap between each SAL centroid and
the network node it was snapped to, which is a proxy for OSM coverage quality at that
location.

In [73]:
# Provincial summary.
snap_cols = ["walk_snap_dist_m", "drive_snap_dist_m"]

for col in snap_cols:
    mode = col.split("_")[0].upper()
    print(f"{mode} SNAP DISTANCES (meters)")
    for prov in ["Gauteng", "KwaZulu-Natal"]:
        vals = all_access.loc[all_access["PR_NAME"] == prov, col].dropna()
        flagged = (vals > SNAP_FLAG_M).sum()
        print(f"  {prov}: median={vals.median():.0f}, mean={vals.mean():.0f}, "
              f"p95={vals.quantile(0.95):.0f}, max={vals.max():.0f}, "
              f"flagged >{SNAP_FLAG_M}m: {flagged} ({flagged / len(vals) * 100:.1f}%)")
    print()

WALK SNAP DISTANCES (meters)
  Gauteng: median=57, mean=83, p95=223, max=25702, flagged >500m: 232 (1.1%)
  KwaZulu-Natal: median=83, mean=226, p95=985, max=9951, flagged >500m: 2149 (12.3%)

DRIVE SNAP DISTANCES (meters)
  Gauteng: median=67, mean=101, p95=290, max=26962, flagged >500m: 398 (1.9%)
  KwaZulu-Natal: median=102, mean=302, p95=1318, max=8435, flagged >500m: 2884 (16.5%)



In [74]:
# Settlement type breakdown of snap flags.
# This confirms where OSM coverage gaps concentrate.
snap_profile = all_access[["EA_CODE", "PR_NAME", "EA_GTYPE",
                           "walk_snap_dist_m", "drive_snap_dist_m"]].copy()

snap_profile["walk_snap_flag"] = snap_profile["walk_snap_dist_m"] > SNAP_FLAG_M
snap_profile["drive_snap_flag"] = snap_profile["drive_snap_dist_m"] > SNAP_FLAG_M

# Snap flag rate by province and settlement type.
for flag_col in ["walk_snap_flag", "drive_snap_flag"]:
    mode = flag_col.split("_")[0].upper()
    print(f"{mode} SNAP FLAG RATE BY SETTLEMENT TYPE")
    summary = (
        snap_profile
        .groupby(["PR_NAME", "EA_GTYPE"])[flag_col]
        .agg(n_flagged = "sum", n_total = "count")
        .assign(pct = lambda x: (x["n_flagged"] / x["n_total"] * 100).round(1))
    )
    print(summary.to_string())
    print()

WALK SNAP FLAG RATE BY SETTLEMENT TYPE
                           n_flagged  n_total     pct
PR_NAME       EA_GTYPE                               
Gauteng       Farms               91      321 28.3000
              Traditional          4      256  1.6000
              Urban              137    20273  0.7000
KwaZulu-Natal Farms              454      758 59.9000
              Traditional       1617     7935 20.4000
              Urban               78     8837  0.9000

DRIVE SNAP FLAG RATE BY SETTLEMENT TYPE
                           n_flagged  n_total     pct
PR_NAME       EA_GTYPE                               
Gauteng       Farms              119      321 37.1000
              Traditional          4      256  1.6000
              Urban              275    20273  1.4000
KwaZulu-Natal Farms              571      758 75.3000
              Traditional       2158     7935 27.2000
              Urban              155     8837  1.8000



**Snap distance takeaway.** OSM coverage gaps concentrate in exactly the SAL
categories that show the worst pharmacy access (Farms and Traditional in KZN).
This means accessibility metrics for those areas are simultaneously the most
policy-relevant and the most uncertain. The snap flag column provides a
per-SAL data quality indicator for downstream consumers.

## SELECT AND MERGE COLUMNS

Build a clean combined table with the minimum columns needed for analysis
and mapping. Avoid carrying forward the full 89-column all_access table.

In [75]:
# AGE COHORT BINNING.
# Convert 2011 age cohort columns from string to numeric, then collapse
# 17 single-cohort columns into five pharmacy-relevant age bands.
# Source columns are dropped after binning; pop_total_2011 is retained
# as a cross-check against sal2011_pop (small differences expected due
# to the population imputation applied in tess_newpred_clean).

AGE_COLS = [
    "F0_4", "F5_9", "F10_14",
    "F15_19", "F20_24", "F25_29", "F30_34",
    "F35_39", "F40_44", "F45_49",
    "F50_54", "F55_59", "F60_64",
    "F65_69", "F70_74", "F75_79", "F80_84", "F85_",
]

# Coerce to numeric, stored as str in the SAL shapefile source.
for col in AGE_COLS:
    all_access[col] = pd.to_numeric(all_access[col], errors = "coerce")

# Build age bands.
all_access["pop_0_14"] = all_access[["F0_4", "F5_9", "F10_14"]].sum(axis = 1, min_count = 1)
all_access["pop_15_34"] = all_access[["F15_19", "F20_24", "F25_29", "F30_34"]].sum(axis = 1, min_count = 1)
all_access["pop_35_49"] = all_access[["F35_39", "F40_44", "F45_49"]].sum(axis = 1, min_count = 1)
all_access["pop_50_64"] = all_access[["F50_54", "F55_59", "F60_64"]].sum(axis = 1, min_count = 1)
all_access["pop_65plus"] = all_access[["F65_69", "F70_74", "F75_79", "F80_84", "F85_"]].sum(axis = 1, min_count = 1)
all_access["pop_total_2011"] = all_access[AGE_COLS].sum(axis = 1, min_count = 1)

# Validation: confirm band totals equal pop_total_2011 within rounding.
band_cols = ["pop_0_14", "pop_15_34", "pop_35_49", "pop_50_64", "pop_65plus"]
recon = all_access[band_cols].sum(axis = 1)
max_discrepancy = (recon - all_access["pop_total_2011"]).abs().max()
print(f"MAX BAND RECONCILIATION DISCREPANCY: {max_discrepancy:.2f} (expect 0.0)")

# Drop the 17 source columns, bands replace them.
all_access = all_access.drop(columns = AGE_COLS)

print(f"AGE BANDS CREATED: {band_cols}")
print(all_access[band_cols + ["pop_total_2011"]].describe().round(1).to_string())

MAX BAND RECONCILIATION DISCREPANCY: 0.00 (expect 0.0)
AGE BANDS CREATED: ['pop_0_14', 'pop_15_34', 'pop_35_49', 'pop_50_64', 'pop_65plus']
        pop_0_14  pop_15_34  pop_35_49  pop_50_64  pop_65plus  pop_total_2011
count 36296.0000 36296.0000 36296.0000 36296.0000  36296.0000      36296.0000
mean    179.3000   263.5000   122.0000    68.9000     31.2000        665.0000
std     114.0000   173.9000    68.6000    40.5000     35.1000        340.0000
min       0.0000     0.0000     0.0000     0.0000      0.0000          0.0000
25%      99.0000   165.0000    83.0000    40.0000     10.0000        473.0000
50%     164.0000   239.0000   117.0000    65.0000     24.0000        633.0000
75%     241.0000   327.0000   153.0000    94.0000     43.0000        816.0000
max    3025.0000  6701.0000  2283.0000   858.0000    932.0000      11717.0000


In [76]:
# COLUMN RENAMES.
# Align source column names from the SAL shapefile to the final output schema.
# Black_Afri and Indian_or are shapefile-truncated names; renamed for clarity.
all_access = all_access.rename(columns = {
    "Black_Afri": "Black_African",
    "Indian_or":  "Indian_Asian",
})

print("COLUMNS RENAMED:")
print(" Black_Afri to Black_African")
print(" Indian_or to Indian_Asian")

COLUMNS RENAMED:
 Black_Afri to Black_African
 Indian_or to Indian_Asian


In [77]:
# EA_TYPE VALUE REPLACEMENT.
# "Formal residential" relabeled to "Suburb" for interpretability.
# The Stats SA label is accurate but not meaningful to non-technical audiences.
# All other EA_TYPE values are retained as-is.
n_replaced = (all_access["EA_TYPE"] == "Formal residential").sum()
all_access["EA_TYPE"] = all_access["EA_TYPE"].replace("Formal residential", "Suburb")

print(f"EA_TYPE REPLACED: 'Formal residential' -> 'Suburb' ({n_replaced:,} rows)")
print()
print("CURRENT EA_TYPE DISTRIBUTION:")
print(all_access["EA_TYPE"].value_counts().to_string())

EA_TYPE REPLACED: 'Formal residential' -> 'Suburb' (12,968 rows)

CURRENT EA_TYPE DISTRIBUTION:
EA_TYPE
Suburb                          12968
Township                         9743
Traditional residential          7108
Informal residential             2823
Vacant                            983
Vacant_*                          900
Farms                             798
Industrial                        723
Commercial                        659
Smallholdings                     650
Collective living quarters        501
Informal residential_*            227
Parks and recreation              135
Formal residential_*              120
Parks and recreation_*             17
Collective living quarters_*       13
Farms_*                             5
Traditional residential_*           4
Industrial_*                        2
Small holdings_*                    1


In [78]:
# COLUMNS FROM ALL_ACCESS (2SFCA SCORES + DEMOGRAPHICS + SNAP DISTANCES).
access_cols = [
    "EA_CODE", "PR_NAME", "EA_GTYPE", "EA_TYPE", "econ_status",
    "WardID", "sal2011_pop", "sal2023_est", "growth_rate", "area_km2",
    "Black_African", "White", "Coloured", "Indian_Asian", "Other",
    "pop_0_14", "pop_15_34", "pop_35_49", "pop_50_64", "pop_65plus",
    "pop_total_2011",
    "num_houses", "num_build",
    "Ai_walk", "Ai_drive",
    "walk_rank", "drive_rank",
    "walk_log", "drive_log",
    "walk_snap_dist_m", "drive_snap_dist_m",
]

# Filter to columns that exist.
access_cols = [c for c in access_cols if c in all_access.columns]
access_df = all_access[access_cols].copy()

print(f"ACCESS TABLE: {access_df.shape}")
print(f"Columns: {list(access_df.columns)}")

ACCESS TABLE: (38380, 31)
Columns: ['EA_CODE', 'PR_NAME', 'EA_GTYPE', 'EA_TYPE', 'econ_status', 'WardID', 'sal2011_pop', 'sal2023_est', 'growth_rate', 'area_km2', 'Black_African', 'White', 'Coloured', 'Indian_Asian', 'Other', 'pop_0_14', 'pop_15_34', 'pop_35_49', 'pop_50_64', 'pop_65plus', 'pop_total_2011', 'num_houses', 'num_build', 'Ai_walk', 'Ai_drive', 'walk_rank', 'drive_rank', 'walk_log', 'drive_log', 'walk_snap_dist_m', 'drive_snap_dist_m']


In [79]:
# COLUMNS FROM PHARMACY_DIST (K-NEAREST DISTANCES).
# Drop columns already in access_df to avoid duplicates on merge.
dist_drop = ["PR_NAME", "WardID", "sal2023_est"]
dist_cols = [c for c in pharmacy_dist.columns if c not in dist_drop]
dist_df = pharmacy_dist[dist_cols].copy()

print(f"DISTANCE TABLE: {dist_df.shape}")
print(f"Columns: {list(dist_df.columns)}")

DISTANCE TABLE: (38380, 23)
Columns: ['EA_CODE', 'centroid_lat', 'centroid_lng', 'euclidean_dist_k1_m', 'euclidean_dist_k1_km', 'euclidean_dist_k2_m', 'euclidean_dist_k2_km', 'euclidean_dist_k3_m', 'euclidean_dist_k3_km', 'walk_dist_k1_m', 'walk_dist_k1_km', 'walk_dist_k2_m', 'walk_dist_k2_km', 'walk_dist_k3_m', 'walk_dist_k3_km', 'drive_dist_k1_m', 'drive_dist_k1_km', 'drive_dist_k2_m', 'drive_dist_k2_km', 'drive_dist_k3_m', 'drive_dist_k3_km', 'walk_circuity_k1', 'drive_circuity_k1']


In [80]:
# COLUMNS FROM THRESHOLD_FLAG (EXCEEDANCE FLAGS ONLY).
# Keep only EA_CODE and the boolean flag columns; distances and
# demographic attributes are already present in the other tables.
flag_cols = ["EA_CODE"] + [c for c in threshold_flag.columns if c.startswith("exceeds_")]
flag_df = threshold_flag[flag_cols].copy()

print(f"FLAG TABLE: {flag_df.shape}")
print(f"Flag columns: {len(flag_cols) - 1}")

FLAG TABLE: (38380, 37)
Flag columns: 36


In [81]:
# MERGE: access + distance + flags on EA_CODE.
combined = (
    access_df
    .merge(dist_df, on = "EA_CODE", how = "inner")
    .merge(flag_df, on = "EA_CODE", how = "inner")
)

print(f"COMBINED TABLE: {combined.shape[0]} rows, {combined.shape[1]} columns")
print(f"Column count breakdown:")
print(f"  From access: {len(access_cols)}")
print(f"  From distance: {len(dist_cols) - 1}")
print(f"  From flags: {len(flag_cols) - 1}")
print(f"  Total: {combined.shape[1]}")

COMBINED TABLE: 38380 rows, 89 columns
Column count breakdown:
  From access: 31
  From distance: 22
  From flags: 36
  Total: 89


## SNAP DISTANCE QUALITY FLAGS

Add boolean columns indicating whether a SAL's centroid snapped more than
500m from a network node. High snap distance means the routing and 2SFCA
results for that SAL carry elevated uncertainty.

In [82]:
# Add quality flag columns.
combined["walk_snap_flag"] = combined["walk_snap_dist_m"] > SNAP_FLAG_M
combined["drive_snap_flag"] = combined["drive_snap_dist_m"] > SNAP_FLAG_M

walk_flagged = combined["walk_snap_flag"].sum()
drive_flagged = combined["drive_snap_flag"].sum()

print(f"WALK SNAP FLAGGED: {walk_flagged} SALs ({walk_flagged / len(combined) * 100:.1f}%)")
print(f"DRIVE SNAP FLAGGED: {drive_flagged} SALs ({drive_flagged / len(combined) * 100:.1f}%)")

WALK SNAP FLAGGED: 2381 SALs (6.2%)
DRIVE SNAP FLAGGED: 3282 SALs (8.6%)


## ACCESS TYPOLOGY

Classify each SAL into one of four quadrant types by crossing the
2SFCA score (Ai) with the k=1 network distance at the policy threshold.
This produces a typology for both walk and drive modes.

| | Distance < threshold | Distance >= threshold |
|---|---|---|
| **Ai = 0** | Connectivity gap | **Pharmacy desert** |
| **Ai > 0, bottom tercile** | Demand overcrowding | Access gap |
| **Ai > 0, top tercile** | Well-served | Artifact zone |

In [83]:
def classify_access(row, ai_col, dist_col, threshold_km, tercile_low, tercile_high):
    """Assign a SAL to one of six access typology categories.

    Categories combine 2SFCA score tiers (zero, low, high) with
    a binary distance threshold to produce interpretable policy labels.
    """
    ai = row[ai_col]
    dist = row[dist_col]

    # Handle NaN distance (unreachable on network).
    if pd.isna(dist):
        if ai == 0:
            return "Pharmacy desert"
        elif ai <= tercile_low:
            return "Access gap"
        else:
            return "Artifact zone"

    near = dist < threshold_km

    if ai == 0 and near:
        return "Connectivity gap"
    elif ai == 0 and not near:
        return "Pharmacy desert"
    elif ai <= tercile_low and near:
        return "Demand overcrowding"
    elif ai <= tercile_low and not near:
        return "Access gap"
    elif near:
        return "Well-served"
    else:
        return "Artifact zone"

In [84]:
# Compute province-specific terciles for Ai scores.
# Only use nonzero Ai values for tercile calculation to avoid
# the zero-inflation skewing the thresholds.
for prov in ["Gauteng", "KwaZulu-Natal"]:
    mask = combined["PR_NAME"] == prov

    for mode, ai_col, dist_col, threshold in [
        ("walk", "Ai_walk", "walk_dist_k1_km", WALK_THRESHOLD_KM),
        ("drive", "Ai_drive", "drive_dist_k1_km", DRIVE_THRESHOLD_KM),
    ]:
        # Tercile boundaries from nonzero Ai values only.
        nonzero_ai = combined.loc[mask & (combined[ai_col] > 0), ai_col]

        if len(nonzero_ai) == 0:
            tercile_low = 0
            tercile_high = 0
        else:
            tercile_low = nonzero_ai.quantile(1 / 3)
            tercile_high = nonzero_ai.quantile(2 / 3)

        typology_col = f"{mode}_typology"

        combined.loc[mask, typology_col] = combined.loc[mask].apply(
            classify_access,
            axis = 1,
            ai_col = ai_col,
            dist_col = dist_col,
            threshold_km = threshold,
            tercile_low = tercile_low,
            tercile_high = tercile_high,
        )

        print(f"{prov} {mode.upper()} TERCILES: low={tercile_low:.4f}, high={tercile_high:.4f}")

print()

Gauteng WALK TERCILES: low=0.0468, high=0.2074
Gauteng DRIVE TERCILES: low=0.0381, high=0.1895
KwaZulu-Natal WALK TERCILES: low=0.0510, high=0.2020
KwaZulu-Natal DRIVE TERCILES: low=0.0384, high=0.1192



In [85]:
# Typology distribution.
for mode in ["walk", "drive"]:
    col = f"{mode}_typology"
    print(f"{mode.upper()} ACCESS TYPOLOGY")
    for prov in ["Gauteng", "KwaZulu-Natal"]:
        prov_df = combined[combined["PR_NAME"] == prov]
        counts = prov_df[col].value_counts()
        pcts = (counts / len(prov_df) * 100).round(1)
        print(f"  {prov}:")
        for cat in ["Well-served", "Demand overcrowding", "Connectivity gap",
                    "Access gap", "Pharmacy desert", "Artifact zone"]:
            n = counts.get(cat, 0)
            p = pcts.get(cat, 0.0)
            print(f"    {cat:25s} {n:6d} ({p:5.1f}%)")
    print()

WALK ACCESS TYPOLOGY
  Gauteng:
    Well-served                 9747 ( 46.7%)
    Demand overcrowding         4874 ( 23.4%)
    Connectivity gap            2795 ( 13.4%)
    Access gap                     0 (  0.0%)
    Pharmacy desert             3434 ( 16.5%)
    Artifact zone                  0 (  0.0%)
  KwaZulu-Natal:
    Well-served                 1462 (  8.3%)
    Demand overcrowding          399 (  2.3%)
    Connectivity gap               0 (  0.0%)
    Access gap                  1853 ( 10.6%)
    Pharmacy desert            10775 ( 61.5%)
    Artifact zone               3041 ( 17.3%)

DRIVE ACCESS TYPOLOGY
  Gauteng:
    Well-served                12780 ( 61.3%)
    Demand overcrowding         6390 ( 30.6%)
    Connectivity gap            1275 (  6.1%)
    Access gap                     0 (  0.0%)
    Pharmacy desert              405 (  1.9%)
    Artifact zone                  0 (  0.0%)
  KwaZulu-Natal:
    Well-served                 5076 ( 29.0%)
    Demand overcrowding   

In [86]:
# Population-weighted typology distribution.
for mode in ["walk", "drive"]:
    col = f"{mode}_typology"
    print(f"{mode.upper()} ACCESS TYPOLOGY (population-weighted)")
    for prov in ["Gauteng", "KwaZulu-Natal"]:
        prov_df = combined[combined["PR_NAME"] == prov]
        total_pop = prov_df["sal2023_est"].sum()
        pop_by_type = prov_df.groupby(col)["sal2023_est"].sum()
        pct_by_type = (pop_by_type / total_pop * 100).round(1)
        print(f"  {prov} (total pop: {total_pop:,.0f}):")
        for cat in ["Well-served", "Demand overcrowding", "Connectivity gap",
                    "Access gap", "Pharmacy desert", "Artifact zone"]:
            pop = pop_by_type.get(cat, 0)
            p = pct_by_type.get(cat, 0.0)
            print(f"    {cat:25s} {pop:12,.0f} ({p:5.1f}%)")
    print()

WALK ACCESS TYPOLOGY (population-weighted)
  Gauteng (total pop: 15,099,415):
    Well-served                  5,686,268 ( 37.7%)
    Demand overcrowding          4,896,139 ( 32.4%)
    Connectivity gap             2,131,873 ( 14.1%)
    Access gap                           0 (  0.0%)
    Pharmacy desert              2,385,135 ( 15.8%)
    Artifact zone                        0 (  0.0%)
  KwaZulu-Natal (total pop: 12,423,893):
    Well-served                    837,726 (  6.7%)
    Demand overcrowding            374,076 (  3.0%)
    Connectivity gap                     0 (  0.0%)
    Access gap                   1,582,909 ( 12.7%)
    Pharmacy desert              7,800,475 ( 62.8%)
    Artifact zone                1,828,708 ( 14.7%)

DRIVE ACCESS TYPOLOGY (population-weighted)
  Gauteng (total pop: 15,099,415):
    Well-served                  7,993,192 ( 52.9%)
    Demand overcrowding          6,151,856 ( 40.7%)
    Connectivity gap               775,106 (  5.1%)
    Access gap       

## SNAP FLAG CROSS-TAB WITH TYPOLOGY

Check whether high-snap SALs disproportionately fall into desert or
connectivity gap categories. If so, these categories carry a data
quality asterisk.

In [87]:
for mode in ["walk", "drive"]:
    typology_col = f"{mode}_typology"
    snap_col = f"{mode}_snap_flag"

    print(f"{mode.upper()} TYPOLOGY x SNAP FLAG")

    cross = (
        combined
        .groupby([typology_col, snap_col])
        .size()
        .unstack(fill_value = 0)
    )

    # Rename columns for readability.
    cross.columns = ["Clean (<= 500m)", f"Flagged (> {SNAP_FLAG_M}m)"]
    cross["Flag Rate"] = (
        cross[f"Flagged (> {SNAP_FLAG_M}m)"] /
        (cross["Clean (<= 500m)"] + cross[f"Flagged (> {SNAP_FLAG_M}m)"]) * 100
    ).round(1)

    print(cross.to_string())
    print()

WALK TYPOLOGY x SNAP FLAG
                     Clean (<= 500m)  Flagged (> 500m)  Flag Rate
walk_typology                                                    
Access gap                      1850                 3     0.2000
Artifact zone                   3021                20     0.7000
Connectivity gap                2771                24     0.9000
Demand overcrowding             5269                 4     0.1000
Pharmacy desert                11902              2307    16.2000
Well-served                    11186                23     0.2000

DRIVE TYPOLOGY x SNAP FLAG
                     Clean (<= 500m)  Flagged (> 500m)  Flag Rate
drive_typology                                                   
Access gap                      2254               179     7.4000
Artifact zone                   2032               120     5.6000
Connectivity gap                1184               119     9.1000
Demand overcrowding             7474                98     1.3000
Pharmacy desert       

## FINAL COLUMN INVENTORY

In [88]:
print(f"FINAL TABLE: {combined.shape[0]} rows, {combined.shape[1]} columns")
print()
print("COLUMN INVENTORY:")
for i, col in enumerate(combined.columns):
    dtype = combined[col].dtype
    n_valid = combined[col].notna().sum()
    print(f"  {i:3d}  {col:40s}  {str(dtype):10s}  {n_valid:6d} non-null")

FINAL TABLE: 38380 rows, 93 columns

COLUMN INVENTORY:
    0  EA_CODE                                   Int64        38380 non-null
    1  PR_NAME                                   str          38380 non-null
    2  EA_GTYPE                                  str          38380 non-null
    3  EA_TYPE                                   str          38380 non-null
    4  econ_status                               str          37091 non-null
    5  WardID                                    int64        38380 non-null
    6  sal2011_pop                               float64      38380 non-null
    7  sal2023_est                               float64      38380 non-null
    8  growth_rate                               float64      37110 non-null
    9  area_km2                                  float64      38380 non-null
   10  Black_African                             float64      36296 non-null
   11  White                                     float64      36296 non-null
   12  Coloured      

In [89]:
# REORGANIZE COLUMNS INTO LOGICAL GROUPS.
# Drops _m duplicates for pharmacy distances (snap distances stay in meters,
# circuity and centroid coords are unaffected). Moves typology and scores
# forward, co-locates demographics, groups snap distances with snap flags.

cols_identifiers = [
    "EA_CODE", "PR_NAME", "WardID",
    "EA_GTYPE", "EA_TYPE", "econ_status",
]

cols_population = [
    "sal2011_pop", "sal2023_est", "growth_rate", "area_km2",
]

cols_demographics = [
    "Black_African", "White", "Coloured", "Indian_Asian", "Other",
    "pop_0_14", "pop_15_34", "pop_35_49", "pop_50_64", "pop_65plus",
    "pop_total_2011",
    "num_houses", "num_build",
]

# Primary analytical outputs first.
cols_primary_outputs = [
    "walk_typology", "drive_typology",
    "Ai_walk", "Ai_drive",
    "walk_rank", "drive_rank",
    "walk_log", "drive_log",
]

# k=1 distances across all modes (reachability).
cols_k1_distances = [
    "walk_dist_k1_km", "drive_dist_k1_km", "euclidean_dist_k1_km",
    "walk_circuity_k1", "drive_circuity_k1",
]

# k=2 and k=3 distances (redundancy and choice).
cols_k2_distances = [
    "walk_dist_k2_km", "drive_dist_k2_km", "euclidean_dist_k2_km",
]
cols_k3_distances = [
    "walk_dist_k3_km", "drive_dist_k3_km", "euclidean_dist_k3_km",
]

# Snap quality: distances and flags together.
cols_snap = [
    "walk_snap_dist_m", "drive_snap_dist_m",
    "walk_snap_flag", "drive_snap_flag",
]

# Threshold exceedance flags grouped by k level.
cols_flags_k1 = sorted([c for c in combined.columns if c.startswith("exceeds_") and "_k1_" in c])
cols_flags_k2 = sorted([c for c in combined.columns if c.startswith("exceeds_") and "_k2_" in c])
cols_flags_k3 = sorted([c for c in combined.columns if c.startswith("exceeds_") and "_k3_" in c])

# Centroid coordinates last for mapping support, not analytical.
cols_centroid = ["centroid_lat", "centroid_lng"]

col_order = (
    cols_identifiers
    + cols_population
    + cols_demographics
    + cols_primary_outputs
    + cols_k1_distances
    + cols_k2_distances
    + cols_k3_distances
    + cols_snap
    + cols_flags_k1
    + cols_flags_k2
    + cols_flags_k3
    + cols_centroid
)

# Filter to columns that actually exist.
col_order = [c for c in col_order if c in combined.columns]

# Identify what was dropped.
dropped = [c for c in combined.columns if c not in col_order]

print(f"COLUMNS RETAINED: {len(col_order)}")
print(f"COLUMNS DROPPED: {len(dropped)}")
for d in dropped:
    print(f"  - {d}")

combined = combined[col_order].copy()
print(f"FINAL SHAPE: {combined.shape}")

COLUMNS RETAINED: 84
COLUMNS DROPPED: 9
  - euclidean_dist_k1_m
  - euclidean_dist_k2_m
  - euclidean_dist_k3_m
  - walk_dist_k1_m
  - walk_dist_k2_m
  - walk_dist_k3_m
  - drive_dist_k1_m
  - drive_dist_k2_m
  - drive_dist_k3_m
FINAL SHAPE: (38380, 84)


## EXPORT CSV

In [90]:
combined.to_csv(CSV_OUTPUT, index = False)
print(f"CSV SAVED: {CSV_OUTPUT}")
print(f"Rows: {len(combined)}, Columns: {combined.shape[1]}")

CSV SAVED: data/combined\sal_combined_access.csv
Rows: 38380, Columns: 84


## EXPORT SHAPEFILE

Join geometry from the SAL dedup shapefile onto the combined table,
then save as a shapefile. Column names are truncated to 10 characters
by the shapefile format.

In [91]:
# Merge geometry onto combined table.
combined_geo = combined.merge(
    sal_geo[["EA_CODE", "geometry"]],
    on = "EA_CODE",
    how = "inner"
)
combined_gdf = gpd.GeoDataFrame(combined_geo, geometry = "geometry", crs = sal_geo.crs)

print(f"GEODATAFRAME: {len(combined_gdf)} features, CRS: {combined_gdf.crs}")

# Save shapefile.
shp_path = os.path.join(SHP_OUTPUT_DIR, "sal_combined_access.shp")
combined_gdf.to_file(shp_path)
print(f"SHAPEFILE SAVED: {shp_path}")

GEODATAFRAME: 38380 features, CRS: EPSG:32735
SHAPEFILE SAVED: data/combined\sal_combined_access_shp\sal_combined_access.shp


## KEY FINDINGS

### The equity gap in three numbers.

**7.8 million people in KZN (62.8% of the province) live in a walking
pharmacy desert** where zero pharmacies are reachable within 3 km on the
pedestrian network. In Gauteng, 2.4 million (15.8%) are in the same
condition. The 47-point gap between provinces is the spatial legacy of
apartheid-era infrastructure investment.

**Even by car, 4.4 million KZN residents (35.8%) remain in a pharmacy
desert** at the 10 km drive threshold. In Gauteng, this drops to
179,000 (1.2%). Driving does not solve KZN's access problem as it looks like it
merely shifts the threshold at which the problem becomes visible.

**The "Connectivity gap" category is almost exclusively a Gauteng
phenomenon** (2,795 SALs walking, 1,275 driving). These are SALs where
a pharmacy exists nearby in Euclidean terms, but the walk or drive
network does not connect the centroid to it. This is a data quality
signal of the OSM coverage gap rather than a genuine access barrier, and it
affects 14.1% of Gauteng's population on the walk network.

### Snap distance validates the uncertainty pattern

**36.3% of "Pharmacy desert" SALs on the drive network have a snap flag**
(centroid >500m from any drive-network node), compared to just 1.1% of
"Well-served" SALs. It looks like the category that matters most for NHI
policy is also the category with the least reliable metrics. Pharmacy desert
classifications for snap-flagged SALs should carry a data quality asterisk
in any policy communication.

### The redundancy deficit

The gap between k=1 and k=3 threshold exceedance rates measures how
many SALs have fragile, single-pharmacy access. In Gauteng, 16.5% of
SALs exceed 3 km walk at k=1, rising to 39.1% at k=3, which is a 22.6-point
redundancy deficit, meaning roughly one in four Gauteng SALs has one
pharmacy within reach but not three. In KZN, the baseline is already
so high (61.3% at k=1) that the deficit is smaller in relative terms
(43.8% at k=3), but the absolute numbers note that over 5.7
million people lack three walking-distance pharmacies.

### Artifact zone interpretation

**17.3% of KZN SALs (1.83 million people) are classified as "Artifact zone"
on the walk network** where nonzero $A_{i}$ scores appear despite being beyond the 3 km
walk threshold. These are mechanically driven by low local demand
(industrial, sparsely populated rural) producing inflated Ai scores.
The walk_typology column correctly flags these so they are not mistaken
for genuinely well-served areas.

## COLUMN DESCRIPTIONS

**82 columns. Organized by analytical priority.**

### Identifiers and geography (cols 0–5)

| Column | Type | Description |
|---|---|---|
| `EA_CODE` | Int64 | Stats SA Enumeration Area code. Unique SAL identifier across all pipeline outputs. |
| `PR_NAME` | str | Province name: Gauteng or KwaZulu-Natal. |
| `WardID` | int64 | 2020 municipal ward code for the ward containing the largest share of this SAL's area. |
| `EA_GTYPE` | str | Settlement geography type: Urban, Traditional, or Farms. |
| `EA_TYPE` | str | Settlement classification: Suburb, Township, Traditional residential, Industrial, Vacant, Informal residential, Small holdings, Collective living quarters, Parks and recreation, Commercial. "Formal residential" renamed to "Suburb" for interpretability. |
| `econ_status` | str | Economic status from 2011 census land classification: Wealthy, Non_Wealthy, or Non_Residential. 1,289 nulls for unclassified SALs. |

### Population (cols 6–9)

| Column | Type | Description |
|---|---|---|
| `sal2011_pop` | float64 | 2011 census population for this SAL. Includes imputed values (houses × 3.3) for SALs with zero census population but nonzero dwelling count (1,190 SALs). |
| `sal2023_est` | float64 | Estimated 2023 population from the proportional step-down model: ward 2023 pop × (SAL 2011 share of ward 2011 pop). |
| `growth_rate` | float64 | Compound annual growth rate 2011–2023. NaN for 1,270 SALs with zero population in both years. |
| `area_km2` | float64 | SAL polygon area in square kilometers, computed from Shape_Area in EPSG:32735. |

### 2011 census demographics (cols 10–22)

2,084 SALs have null values across all demographic columns (non-residential or
unmatched SALs absent from the 2011 census release).

| Column | Type | Description |
|---|---|---|
| `Black_African` | float64 | 2011 census count of Black African population. Renamed from shapefile-truncated `Black_Afri`. |
| `White` | float64 | 2011 census count of White population. |
| `Coloured` | float64 | 2011 census count of Coloured population. |
| `Indian_Asian` | float64 | 2011 census count of Indian or Asian population. The census does not disaggregate these groups; this is a combined category. Renamed from shapefile-truncated `Indian_or`. |
| `Other` | float64 | 2011 census count of Other population group. |
| `pop_0_14` | float64 | 2011 census population aged 0–14. Pediatric, caregiver is the pharmacy user. Derived by summing F0_4 + F5_9 + F10_14. |
| `pop_15_34` | float64 | Population aged 15–34. Young adult baseline demand. Derived from F15_19 + F20_24 + F25_29 + F30_34. |
| `pop_35_49` | float64 | Population aged 35–49. Chronic disease onset, rising prescription dependency. |
| `pop_50_64` | float64 | Population aged 50–64. Highest working-age pharmacy dependency. |
| `pop_65plus` | float64 | Population aged 65+. Highest medication need and greatest travel difficulty. |
| `pop_total_2011` | float64 | Sum of all 17 original age cohorts. Cross-check against `sal2011_pop`; diverges for 1,190 imputed SALs where `sal2011_pop` includes the houses × 3.3 adjustment but age cohorts do not. |
| `num_houses` | float64 | 2011 census dwelling count for this SAL. Used in the zero-population imputation step. |
| `num_build` | float64 | Google Open Buildings v3 building footprint count. Foundation for the dasymetric weighting exploration (not yet integrated into the population model). |

### Primary outputs: typology, 2SFCA scores, and transforms (cols 23–30)

| Column | Type | Description |
|---|---|---|
| `walk_typology` | str | Six-category access classification combining Ai_walk with walk_dist_k1_km at the 3 km policy threshold. See typology definitions below. |
| `drive_typology` | str | Same classification for drive mode at the 10 km policy threshold. |
| `Ai_walk` | float64 | Two-Step Floating Catchment Area accessibility score on the pedestrian network. Higher = better access. Zero means no pharmacy reachable within the catchment radius. Uses negative exponential decay (β = 0.0003) and province-specific catchments (Gauteng: 2 km, KZN: 3 km). |
| `Ai_drive` | float64 | 2SFCA accessibility score on the drive network. Same decay function, with catchments of 5 km (Gauteng) and 10 km (KZN). |
| `walk_rank` | float64 | Province-specific percentile rank of Ai_walk (0–1). More interpretable than raw Ai for communication and choropleth mapping. |
| `drive_rank` | float64 | Province-specific percentile rank of Ai_drive (0–1). |
| `walk_log` | float64 | Natural log transform of Ai_walk. Compresses the right tail for visualization; zero-Ai SALs remain at zero. |
| `drive_log` | float64 | Natural log transform of Ai_drive. |

**Typology definitions:**

| Category | $A_i$ score | Distance | Interpretation |
|---|---|---|---|
| Well-served | Top tercile (nonzero) | < threshold | Pharmacy nearby with favorable supply-to-demand ratio. |
| Demand overcrowding | Bottom tercile (nonzero) | < threshold | Pharmacy nearby but serving too large a population. Candidate for NHI capacity investment at the existing site. |
| Connectivity gap | Zero | < threshold | Pharmacy exists nearby in Euclidean terms but is network-unreachable. Likely an OSM coverage gap or catchment boundary artifact. |
| Access gap | Bottom tercile (nonzero) | ≥ threshold | Far from a pharmacy, and that pharmacy is also under-supplied relative to demand. Secondary infrastructure target. |
| Pharmacy desert | Zero | ≥ threshold | No pharmacy reachable on the network and nearest pharmacy is also far in absolute terms. Primary NHI infrastructure target. |
| Artifact zone | Top tercile (nonzero) | ≥ threshold | High $A_i$ mechanically driven by low local demand (industrial, sparsely populated rural). Score is not policy-relevant; use distance columns as primary indicator. |

Terciles are computed province-specifically from nonzero $A_i$ values only, preventing
zero-inflation from collapsing the boundary toward zero. Walk and drive terciles are
computed independently.

### k=1 distances and circuity (cols 31–35)

k=1 answers the reachability question: can any pharmacy be reached at all?

| Column | Type | Description |
|---|---|---|
| `walk_dist_k1_km` | float64 | Shortest-path distance (km) on the OSM pedestrian network to the nearest pharmacy, via Dijkstra. NaN if no path found within the 50 km cutoff (4,931 SALs, 12.8%). |
| `drive_dist_k1_km` | float64 | Same on the OSM drive network. Graph converted to undirected for routing. NaN for 5,001 SALs (13.0%). |
| `euclidean_dist_k1_km` | float64 | Straight-line distance (km) to nearest pharmacy. Always non-null; used as a lower bound and baseline for circuity. |
| `walk_circuity_k1` | float64 | walk_dist_k1_km / euclidean_dist_k1_km. Values of 1.2–1.5 are typical urban grids; >3 suggests network gaps or major barriers. |
| `drive_circuity_k1` | float64 | Same ratio for the drive network. |

### k=2 distances (cols 36–38)

k=2 answers the redundancy question: is there a fallback pharmacy?

| Column | Type | Description |
|---|---|---|
| `walk_dist_k2_km` | float64 | Walk network distance (km) to the 2nd nearest pharmacy. NaN for 8,681 SALs (22.6%). |
| `drive_dist_k2_km` | float64 | Drive network distance (km) to the 2nd nearest pharmacy. NaN for 8,660 SALs (22.6%). |
| `euclidean_dist_k2_km` | float64 | Straight-line distance (km) to the 2nd nearest pharmacy. Always non-null. |

### k=3 distances (cols 39–41)

k=3 answers the choice question: do residents have meaningful pharmacy options?

| Column | Type | Description |
|---|---|---|
| `walk_dist_k3_km` | float64 | Walk network distance (km) to the 3rd nearest pharmacy. NaN for 9,689 SALs (25.2%). |
| `drive_dist_k3_km` | float64 | Drive network distance (km) to the 3rd nearest pharmacy. NaN for 9,679 SALs (25.2%). |
| `euclidean_dist_k3_km` | float64 | Straight-line distance (km) to the 3rd nearest pharmacy. Always non-null. |

### Snap distances and data quality flags (cols 42–45)

| Column | Type | Description |
|---|---|---|
| `walk_snap_dist_m` | float64 | Euclidean distance (meters) from the SAL centroid to the nearest pedestrian network node. Measures how far the Dijkstra origin was displaced from the true centroid. |
| `drive_snap_dist_m` | float64 | Same for the drive network. Values differ because walk and drive networks have different node sets. |
| `walk_snap_flag` | bool | True if walk_snap_dist_m > 500m. Scores for flagged SALs carry elevated measurement uncertainty. |
| `drive_snap_flag` | bool | True if drive_snap_dist_m > 500m. |

### Threshold exceedance flags w/ k=1 (cols 46–57)

| Column pattern | Type | Description |
|---|---|---|
| `exceeds_walk_k1_{1,2,3,5}km` | bool | True if nearest pharmacy exceeds the walk threshold. NaN walk distances treated as exceeding. |
| `exceeds_drive_k1_{5,10,15,20}km` | bool | True if nearest pharmacy exceeds the drive threshold. |
| `exceeds_euclidean_k1_{1,3,5,10}km` | bool | True if straight-line distance to nearest pharmacy exceeds the threshold. Always non-null. |

### Threshold exceedance flags w/ k=2 (cols 58–69)

Same structure as k=1, applied to the 2nd nearest pharmacy. The gap between
k=1 and k=2 exceedance rates at the same threshold is the **redundancy deficit**.

### Threshold exceedance flags w/ k=3 (cols 70–81)

Same structure, applied to the 3rd nearest pharmacy. k=3 exceedance measures
whether a SAL has meaningful pharmacy choice or depends on a single facility.

### Centroid coordinates (cols 82–83)

| Column | Type | Description |
|---|---|---|
| `centroid_lat` | float64 | Latitude of SAL geometric centroid (WGS84 / EPSG:4326). |
| `centroid_lng` | float64 | Longitude of SAL geometric centroid (WGS84 / EPSG:4326). |

## COLUMNS DROPPED AND EXCLUDED

### Dropped during reorganization (9 columns: 91 to 82)

All dropped columns are redundant unit conversions. Each has a kilometers
counterpart retained in the table.

| Dropped | Retained equivalent |
|---|---|
| `euclidean_dist_k{1,2,3}_m` | `euclidean_dist_k{1,2,3}_km` |
| `walk_dist_k{1,2,3}_m` | `walk_dist_k{1,2,3}_km` |
| `drive_dist_k{1,2,3}_m` | `drive_dist_k{1,2,3}_km` |

Snap distance columns (`walk_snap_dist_m`, `drive_snap_dist_m`) are retained
in meters because the 500m flag threshold requires sub-kilometer resolution.

### Excluded during initial column selection (never carried forward)

These columns existed in the 91-column `tess_all_access_w_snap.csv` source
but were not included in `access_cols`. They are recoverable by joining on
`EA_CODE` to the indicated source file.

| Column(s) | Category | Why excluded |
|---|---|---|
| `SP_CODE`, `SP_NAME`, `MP_CODE`, `MP_NAME`, `MN_MDB_C`, `MN_CODE`, `MN_NAME`, `MN_TYPE`, `DC_MDB_C`, `DC_MN_C`, `DC_NAME`, `PR_MDB_C`, `PR_CODE`, `MD_CODE`, `MD_NAME` | Geographic hierarchy | Sub-place through district council codes. Province captured in `PR_NAME`, ward in `WardID`. Recoverable from SAL shapefile. |
| `SAL_CODE` | Alternate ID | Redundant with `EA_CODE`. |
| `OBJECTID_1`, `OBJECTID_2`, `OBJECTID_3` | Join artifacts | ArcGIS auto-generated row IDs. |
| `EA_CODE_1`, `EA_CODE_12`, `EA_CODE_13` | Join artifacts | Duplicate EA_CODE columns from successive spatial joins. |
| `census_war` | Join artifact | Shapefile-truncated ward code; clean version is `WardID`. |
| `AREA`, `PERCENTAGE`, `FREQUENCY`, `MAX_AREA` | Join artifacts | Intermediate spatial join metrics. Ward assignment captured in `WardID`. |
| `sal2011_po`, `sal_pop_de` | Truncated duplicates | `sal2011_po` is the DBF-truncated `sal2011_pop`. `sal_pop_de` is raw density, re-derivable as `sal2011_pop / area_km2`. |
| `Shape_Leng`, `Shape_Le_1`, `Shape_Area` | Raw geometry fields | `area_km2` is the clean derived equivalent. |
| `ALBERS_ARE`, `EA_area_km` | Redundant area | Alternative area calculations superseded by `area_km2`. |
| `F4_class`, `F4_class_2` | Renamed source | Original column names for `econ_status`. |
| `old_EA_TYP` | Deprecated label | Legacy EA type classification superseded by `EA_TYPE`. |
| `url` | Reference link | Census 2011 Adrian Frith lookup URL. Not analytical. |
| `smallplace` | Place label | Descriptive small-place name. Recoverable from SAL shapefile. |
| `population` | Alternate pop count | 2011 total from original census shapefile (pre-imputation). Cross-checkable via `pop_total_2011`. |
| `ward2023_pop` | Model intermediate | 2023 ward total (step-down numerator). Source: `pop_pred_final.csv`. |
| `ward2011_sum` | Model intermediate | Sum of 2011 SAL pops within ward (step-down denominator). Source: `pop_pred_final.csv`. |
| `share2011`, `dasym_weight` | Model intermediate | Identical columns, SAL share of ward 2011 pop. Source: `pop_pred_final.csv`. |
| `sal_dense`, `log_density` | EDA variables | Population density and log transform from the step-down notebook. |
| `node` | Routing debug | OSM network node ID for centroid snap. Useful for reproducing Dijkstra runs only. |
| `F0_4` through `F85_` (17 cols) | Collapsed into age bands | Replaced by `pop_0_14`, `pop_15_34`, `pop_35_49`, `pop_50_64`, `pop_65plus`. |

## NOTES AND LIMITATIONS

### Data

This table merges outputs from three upstream notebooks. Each row
represents one Small Area Layer (SAL) in Gauteng or KwaZulu-Natal,
the smallest census geography published by Stats SA.

- **2SFCA scores** (Ai_walk, Ai_drive) were computed by
  `tess_2sfca_clean.ipynb` using negative exponential decay
  (β = 0.0003) with province-specific catchment distances.
- **k-nearest distances** were computed by
  `sal_pharmacy_distance_k3.ipynb` using single-source Dijkstra
  from each pharmacy node with a 50 km cutoff.
- **Threshold flags** were computed by `network_threshold.ipynb`
  by applying binary thresholds to the k-nearest distances.
- **Snap distances** were computed by `snap_distance_append.ipynb`
  using `ox.nearest_nodes()` to measure the centroid-to-node gap.

All spatial computations use EPSG:32735 (UTM zone 35S, meters).
Pharmacy coordinates originate in WGS84 and are projected for
distance calculations. The SAL polygon source is `sal_w_ward_dedup.shp`
(38,380 features after deduplication from Jill's ArcGIS spatial join).

### Population model assumptions

The `sal2023_est` column is a modeled estimate, not a census count.
It uses a proportional step-down method: each SAL receives a share
of its parent ward's 2023 population proportional to its share of
the ward's 2011 census population. This assumes that the within-ward
population distribution has not changed since 2011, an assumption
that breaks for SALs with significant new development (greenfield
housing, informal settlement growth) or depopulation. The 1,270 SALs
with zero estimated population (truly vacant in 2011 with no dwellings)
are included in SAL-count statistics but contribute nothing to
population-weighted statistics.

### Centroid placement

All distances are measured from geometric (unweighted) SAL centroids.
For compact urban SALs, this closely approximates the population
center. For large, irregular rural SALs (common in KZN Traditional
areas), the geometric centroid may fall in uninhabited terrain, a
river valley, hilltop, or nature reserve, introducing systematic
measurement error. Population-weighted centroids using Google Open
Buildings v3 footprints would reduce this bias but are not yet
integrated into the pipeline.

### Network coverage and snap distance

OSM pedestrian and drive networks have uneven coverage. Urban areas
are well-mapped; rural KZN and informal settlements have significant
gaps. The `walk_snap_dist_m` and `drive_snap_dist_m` columns quantify
this gap for each SAL. When a centroid snaps to a node 500m+ away, the
Dijkstra routing started from a displaced origin, and the resulting
Ai score and network distance carry proportional uncertainty.

The snap flag cross-tab with access typology shows that 36.3% of
drive-network "Pharmacy desert" SALs are snap-flagged, compared to
1.1% of "Well-served" SALs. This concentration means that some
pharmacy desert classifications may reflect OSM road data gaps
rather than genuine pharmacy absence. However, areas with poor OSM
coverage are also areas with poor road infrastructure, so the two
effects reinforce rather than contradict each other.

### NaN network distances

NaN values in `walk_dist_k{1,2,3}_km` and `drive_dist_k{1,2,3}_km`
indicate that no path exists between the SAL centroid and the k-th
pharmacy within the 50 km Dijkstra cutoff. This affects 4,931 SALs
(12.8%) at walk k=1 and 5,001 SALs (13.0%) at drive k=1. NaN
distances are treated as exceeding all thresholds in the exceedance
flags. In the access typology, NaN distances are treated as "beyond
threshold."

### 2SFCA methodology

The 2SFCA implementation uses a single supply unit per pharmacy
($S_{j} = 1$), meaning all pharmacies are assumed to have equal capacity.
In practice, a high-volume chain pharmacy in Sandton and a single-
pharmacist dispensary in rural KZN are treated identically. Pharmacy
capacity data (staffing, dispensing volume, hours) are not available
in the dataset.

The minimum weighted population floor (`MIN_WEIGHTED_POP = 50`)
prevents near-infinite $R_{j}$ ratios from pharmacies with extremely small
catchment populations. This caps the maximum $R_{j}$ at 20 (i.e., 1000/50).
The floor value is arbitrary; sensitivity to alternative values (25,100) has not been tested.

### Catchment thresholds

Province-specific catchment distances (Gauteng walk: 2 km, KZN walk:
3 km; Gauteng drive: 5 km, KZN drive: 10 km) create a discontinuity
at the province boundary. SALs on the KZN side of the border receive
larger catchments than adjacent SALs in Gauteng's East Rand, despite
comparable urbanization. A variable-catchment approach that assigns
thresholds by settlement type (EA_GTYPE) rather than province would
eliminate this boundary effect but has not been implemented. The
commented-out variable catchment code exists in `tess_2sfca_clean.ipynb`.

### Access typology terciles

The typology uses province-specific terciles computed from nonzero
Ai values only. This prevents the large number of zero-Ai SALs
(53.2% of Gauteng walk, 74.3% of KZN walk) from pulling the
tercile boundaries toward zero. The tercile approach means that
"low" and "high" are relative within each province, so a "high"
Ai in KZN could be numerically lower than a "low" Ai in Gauteng.
Cross-province comparison of Ai scores requires caution.

### Transport mode coverage

The walk network captures sidewalks, footpaths, and pedestrian-
accessible roads mapped in OSM. The drive network captures roads
tagged for motor vehicle access. Neither mode captures minibus taxi
routes, the dominant public transport mode in South Africa. A SAL
classified as exceeding the walk threshold may be well-served by
a taxi route passing through a pharmacy cluster. Taxi route data
are not available at sufficient spatial resolution for integration.

### Threshold arbitrariness

Policy thresholds (3 km walk, 10 km drive) are reference points, not
natural discontinuities. A SAL at 3.1 km is classified differently from
one at 2.9 km despite near-identical access. The multi-threshold
approach (1/2/3/5 km walk, 5/10/15/20 km drive) at three k levels
mitigates this by showing the full exceedance curve rather than a
single binary cut. The threshold flags should be used as a family,
not individually.

### Pharmacy data coverage

The pharmacy dataset (2,241 geocoded records: 1,453 Gauteng, 699 KZN)
combines records from GEMS, Momentum, Wooltru, SAMWUMED, and Vitality.
It covers private-sector and government-employee pharmacies. Hospital
pharmacies and informal dispensing points (clinic dispensaries, mobile
units) are not included. The actual number of points where medication
can be obtained is likely higher than the pharmacy count, meaning the
analysis presents a conservative (pessimistic) view of access.

### Shapefile column truncation

The `.shp` export truncates column names to 10 characters due to the
DBF format limitation. Use the CSV export for any analysis that
references column names programmatically. The shapefile is intended
for GIS visualization only.